<div style="background:#0b2545; padding:25px; border-radius:10px; color:white;">

<h2 style="margin-bottom:5px;">Balancing Transparency and Predictive Power:</h2>
<h4 style="margin-top:0; font-weight:normal;">
A Multi-Objective Framework for Explainable Crop Recommendation Systems
</h4>

<p><b>Submitted in partial fulfilment of the requirements for the degree of</b><br>
Master of Science in Data Science</p>

<p><b>Research Type:</b> Quantitative · Machine Learning · Explainable AI (XAI)<br>
<b>Dataset:</b> Crop Recommendation Dataset<br>
<b>Tools:</b> Python · scikit-learn · XGBoost · SHAP · LIME · Plotly</p>

</div>

<br>

### **Abstract**

This study presents a reproducible, end-to-end analytical pipeline for the development of an explainable crop recommendation system. Four machine learning classifiers, Random Forest, XGBoost, Support Vector Machine, and Artificial Neural Network—are rigorously evaluated based on both predictive performance and a structured explainability framework. 

To address the inherent trade-off between accuracy and interpretability, a novel *Multi-Objective Composite Score (MOCS)* is proposed, enabling a principled quantification of model performance across competing objectives. The framework supports informed model selection for farmer-centric advisory systems, where both trustworthiness and predictive precision are critical for real-world deployment.

---
## Table of Contents
| § | Section | Description |
|:-:|:--------|:------------|
| 1 | [Environment Setup & Configuration](#sec1) | Libraries, seeds, colour palettes, feature maps |
| 2 | [Exploratory Data Analysis](#sec2) | Distribution, correlation, PCA, feature engineering |
| 3 | [Pre-processing & Feature Engineering](#sec3) | Encoding, scaling, train/test split |
| 4 | [Multi-Objective Model Training](#sec4) | RF · XGBoost · SVM · ANN |
| 5 | [Comparative Performance Evaluation](#sec5) | Metrics table, bar chart, cross-validation |
| 6 | [Global Explainability — SHAP](#sec6) | TreeExplainer, permutation importance, consensus plot |
| 7 | [Local Explainability — LIME](#sec7) | Instance explanations, cross-model stability |
| 8 | [Multi-Objective Trade-off Analysis](#sec8) | XAI rubric, MOCS, Pareto bubble chart |
| 9 | [Statistical Significance Testing](#sec9) | Friedman test, Wilcoxon pairs, Cohen's *d* |
| 10 | [Conclusions & Summary Dashboard](#sec10) | Narrative summary, final 6-panel dashboard |

---
<a id="sec1"></a>
### Section 1 — Environment Setup & Configuration

This section initialises all third-party libraries, establishes a global random seed for
reproducibility, and defines shared aesthetic constants (colour palettes, feature label maps,
and the Plotly template) used consistently throughout the analysis.

In [2]:
import numpy as np
import pandas as pd
import warnings, time, json
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from itertools import combinations
from scipy import stats

from xgboost import XGBClassifier
import shap
from lime import lime_tabular

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Aesthetic Constants
MODEL_COLORS = {
    'Random Forest': '#2ecc71',
    'XGBoost':       '#3498db',
    'SVM':           '#e67e22',
    'ANN':           '#9b59b6'
}

CAT_COLORS = {
    'Cereals':               '#F39C12',
    'Pulses':                '#27AE60',
    'Fruits':                '#E74C3C',
    'Melons & Cucurbits':    '#16A085',
    'Plantation Crops':      '#8E44AD',
    'Fiber & Industrial':    '#34495E'
}

PLOTLY_TEMPLATE = 'plotly_white'

# Agronomic Groupings
groups = {
    'Cereals':            ['rice', 'maize'],
    'Pulses':             ['chickpea', 'kidneybeans', 'pigeonpeas',
                           'mothbeans', 'mungbean', 'blackgram', 'lentil'],
    'Fruits':             ['banana', 'mango', 'apple', 'orange',
                           'papaya', 'pomegranate', 'grapes'],
    'Melons & Cucurbits': ['watermelon', 'muskmelon'],
    'Plantation Crops':   ['coconut', 'coffee'],
    'Fiber & Industrial': ['cotton', 'jute']
}

crop_to_cat = {crop: cat for cat, crops in groups.items() for crop in crops}
cat_names   = list(groups.keys())

# Feature Definitions
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

feature_labels = {
    'N':           'Nitrogen (N)',
    'P':           'Phosphorus (P)',
    'K':           'Potassium (K)',
    'temperature': 'Temperature (°C)',
    'humidity':    'Humidity (%)',
    'ph':          'Soil pH',
    'rainfall':    'Rainfall (mm)'
}

print("Environment initialised successfully.")
print(f"  Plotly {__import__('plotly').__version__} | SHAP {shap.__version__}")

Environment initialised successfully.
  Plotly 6.3.0 | SHAP 0.51.0


---
<a id="sec2"></a>
### Exploratory Data Analysis

The exploratory phase characterises the dataset's structure, class balance, and
inter-feature relationships. Seven agrochemical and climatic variables serve as
predictors for 22 crop classes, which are further grouped into six agronomic
categories to facilitate higher-level pattern recognition.

#### Dataset Overview

In [3]:
# Load and annotate the dataset with agronomic category labels
df = pd.read_csv('Crop_recommendation.csv')
df['category'] = df['label'].map(crop_to_cat)

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"  Rows           : {df.shape[0]:,}")
print(f"  Features       : {df.shape[1] - 2}  (excl. label & category)")
print(f"  Crop classes   : {df['label'].nunique()}")
print(f"  Categories     : {df['category'].nunique()}")
print(f"  Missing values : {df.isnull().sum().sum()}")
print(f"  Duplicates     : {df.duplicated().sum()}")
print()
display(df.describe().round(3))

DATASET OVERVIEW
  Rows           : 2,200
  Features       : 7  (excl. label & category)
  Crop classes   : 22
  Categories     : 6
  Missing values : 0
  Duplicates     : 0



,N,P,K,temperature,humidity,ph,rainfall
count,2200.000,2200.000,2200.000,2200.000,2200.000,2200.000,2200.000
mean,50.552,53.363,48.149,25.616,71.482,6.469,103.464
std,36.917,32.986,50.648,5.064,22.264,0.774,54.958
min,0.000,5.000,5.000,8.826,14.258,3.505,20.211
25%,21.000,28.000,20.000,22.769,60.262,5.972,64.552
50%,37.000,51.000,32.000,25.599,80.473,6.425,94.868
75%,84.250,68.000,49.000,28.562,89.949,6.924,124.268
max,140.000,145.000,205.000,43.675,99.982,9.935,298.560


#### Class and Category Distribution

In [4]:
# relative frequency of each agronomic category
fig = make_subplots(
    rows=1, cols=1,
    subplot_titles=('Crop Category Distribution',),
    specs=[[{'type': 'pie'}]]
)

cat_counts = df['category'].value_counts()

fig.add_trace(
    go.Pie(
        labels=cat_counts.index,
        values=cat_counts.values,
        marker=dict(
            colors=[CAT_COLORS[c] for c in cat_counts.index],
            line=dict(color='white', width=3)
        ),
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Samples: %{value}<br>%{percent}<extra></extra>',
        pull=[0.05]*len(cat_counts)
    ),
    row=1, col=1
)

fig.update_layout(
    title=dict(
        text='<b>Dataset Distribution by Crop Category</b>',
        x=0.5, font=dict(size=18)
    ),
    template=PLOTLY_TEMPLATE,
    height=500,
    showlegend=False
)
fig.show()

crop_counts = df['label'].value_counts().reset_index()
crop_counts.columns = ['Crop', 'Count']
crop_counts

,Crop,Count
0,rice,100
1,maize,100
2,jute,100
3,cotton,100
4,coconut,100
5,papaya,100
6,orange,100
7,apple,100
8,muskmelon,100
9,watermelon,100


#### Feature Distributions Across Crop Categories

In [5]:
# Box plots of all seven predictors stratified by agronomic category
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[feature_labels[f] for f in features],
    vertical_spacing=0.18,
    horizontal_spacing=0.06
)

positions = [(r+1, c+1) for r in range(2) for c in range(4)]

for i, feat in enumerate(features):
    r, c = positions[i]
    for cat in cat_names:
        fig.add_trace(
            go.Box(
                y=df[df['category'] == cat][feat],
                name=cat,
                marker=dict(color=CAT_COLORS[cat]),
                line=dict(width=1.2),
                boxpoints=False,
                showlegend=False,
                hoverinfo='skip'
            ),
            row=r, col=c
        )

fig.update_layout(
    title=dict(
        text='<b>Distribution of Key Features Across Crop Categories</b>',
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white',
    font=dict(family='Arial', size=11, color='black'),
    height=750, width=1100, showlegend=False,
    margin=dict(l=50, r=30, t=80, b=50)
)
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')
fig.show()

#### Feature Correlation Matrix

In [6]:
# Pearson correlation heatmap across all predictor variables
corr = df[features].corr().round(3)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=[feature_labels[f] for f in features],
    y=[feature_labels[f] for f in features],
    colorscale='RdYlGn', zmid=0, zmin=-1, zmax=1,
    text=corr.round(2).values,
    texttemplate='%{text}', textfont=dict(size=10),
    hovertemplate='<b>%{y}</b> vs <b>%{x}</b><br>Pearson r = %{z:.3f}<extra></extra>',
    colorbar=dict(title=dict(text='Pearson r', side='right'), thickness=15)
))

fig.update_layout(
    title=dict(
        text='<b>Feature Correlation Matrix</b>',
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white',
    font=dict(family='Arial', size=11, color='black'),
    height=550, width=700,
    margin=dict(l=80, r=30, t=80, b=80)
)
fig.update_xaxes(tickangle=-30, showgrid=False, tickfont=dict(size=10))
fig.update_yaxes(showgrid=False, tickfont=dict(size=10))
fig.show()

#### Principal Component Analysis (PCA) — 2D Feature Space

In [7]:
# PCA projection to assess linear separability across categories
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(df[features])

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_pca)

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1],
    'Category': df['category'], 'Crop': df['label']
})

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
total_var = pc1_var + pc2_var

fig = px.scatter(
    pca_df, x='PC1', y='PC2',
    color='Category', symbol='Category',
    color_discrete_map=CAT_COLORS,
    hover_data={'Crop': True, 'PC1': ':.2f', 'PC2': ':.2f'},
    opacity=0.75,
    title=(
        f"<b>PCA Projection of Feature Space</b><br>"
        f"<span style='font-size:12px'>"
        f"PC1 ({pc1_var:.1f}% variance), PC2 ({pc2_var:.1f}% variance) — "
        f"Total: {total_var:.1f}%</span>"
    ),
    template='simple_white', height=550, width=800
)

fig.update_traces(marker=dict(size=6, line=dict(width=0.5, color='white')))
fig.update_layout(
    title=dict(x=0.5, xanchor='center', font=dict(size=18, family='Arial')),
    font=dict(family='Arial', size=11, color='black'),
    legend=dict(title='Crop Category', orientation='h', x=0.5, y=-0.15, xanchor='center'),
    margin=dict(l=50, r=30, t=90, b=70)
)
fig.show()

#### Per-Feature Violin Plots — Top Crops

In [8]:
# Violin plots comparing rainfall distribution across the six most frequent crops
def plot_feature_kde_overlap(df, feature, top_n_crops=5):
    crops = df['label'].value_counts().nlargest(top_n_crops).index
    fig = go.Figure()

    for i, crop in enumerate(crops):
        vals = df[df['label'] == crop][feature].dropna()
        fig.add_trace(go.Violin(
            y=vals, name=crop, box_visible=True, meanline_visible=True,
            opacity=0.75,
            marker=dict(
                color=px.colors.qualitative.Plotly[i % 10],
                line=dict(width=0.5, color='white')
            )
        ))

    fig.update_layout(
        title=dict(
            text=(
                f"<b>{feature_labels[feature]} Distribution Across Major Crops</b><br>"
                f"<span style='font-size:12px'>Top {top_n_crops} crops by sample frequency</span>"
            ),
            x=0.5, xanchor='center', font=dict(size=18, family='Arial')
        ),
        template='simple_white', height=450, width=800,
        font=dict(family='Arial', size=11),
        xaxis_title='Crop', yaxis_title=feature_labels[feature],
        legend=dict(orientation='h', x=0.5, y=-0.2, xanchor='center'),
        margin=dict(l=60, r=30, t=90, b=80)
    )
    fig.show()

plot_feature_kde_overlap(df, 'rainfall', top_n_crops=6)

#### Intra-Category Correlation Patterns

In [9]:
# Per-category Pearson correlation heatmaps to reveal crop-specific interactions
def plot_class_correlations(df, features, categories, n_cols=3):
    n_cats = len(categories)
    n_rows = (n_cats + n_cols - 1) // n_cols
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=categories)

    for idx, cat in enumerate(categories):
        subset = df[df['category'] == cat][features]
        corr   = subset.corr().round(2)
        r, c   = divmod(idx, n_cols)

        fig.add_trace(
            go.Heatmap(
                z=corr.values, x=features, y=features,
                colorscale='RdBu_r', zmid=0, showscale=False,
                text=corr.values, texttemplate='%{text}', textfont=dict(size=9)
            ),
            row=r+1, col=c+1
        )

    fig.update_layout(
        title=dict(
            text=(
                "<b>Feature Correlation Patterns by Crop Category</b><br>"
                "<span style='font-size:12px'>Pearson correlations within each category</span>"
            ),
            x=0.5, xanchor='center', font=dict(size=18, family='Arial')
        ),
        template='simple_white', height=420*n_rows, width=900,
        font=dict(family='Arial', size=10),
        margin=dict(l=50, r=30, t=90, b=50)
    )
    fig.show()

plot_class_correlations(df, features, cat_names)

#### Soil Fertility Index

In [10]:
# Derived composite nutrient index: SFI = (N + P + K) / 3
df['soil_fertility_index'] = (df['N'] + df['P'] + df['K']) / 3
df_derived = df.copy()

fig = px.box(
    df_derived, x='category', y='soil_fertility_index',
    color='category', color_discrete_map=CAT_COLORS,
    template='simple_white',
    labels={'soil_fertility_index': 'Soil Fertility Index'},
    height=450, width=800
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Soil Fertility Index Across Crop Categories</b><br>"
            "<span style='font-size:12px'>(N + P + K) / 3 composite nutrient index</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    font=dict(family='Arial', size=11),
    legend=dict(orientation='h', x=0.5, y=-0.2, xanchor='center'),
    margin=dict(l=60, r=40, t=90, b=80)
)
fig.show()

#### Preliminary Permutation Feature Importance

In [11]:
# Quick Random Forest fit on scaled data to compute permutation importance
from sklearn.inspection import permutation_importance

_rf_eda  = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
_le_eda  = LabelEncoder()
_y_eda   = _le_eda.fit_transform(df['label'])
_sc_eda  = StandardScaler()
_Xs_eda  = _sc_eda.fit_transform(df[features])

_rf_eda.fit(_Xs_eda, _y_eda)

perm_eda = permutation_importance(_rf_eda, _Xs_eda, _y_eda,
                                   n_repeats=10, random_state=SEED, n_jobs=1)

imp_df = pd.DataFrame({
    'Feature':    [feature_labels[f] for f in features],
    'Importance': perm_eda.importances_mean,
    'Std':        perm_eda.importances_std
}).sort_values('Importance', ascending=True)

fig = px.bar(
    imp_df, x='Importance', y='Feature', orientation='h', error_x='Std',
    color='Importance', color_continuous_scale='Viridis',
    template='simple_white', height=450, width=850
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Permutation Feature Importance (Random Forest — EDA)</b><br>"
            "<span style='font-size:12px'>Higher values indicate stronger predictive contribution</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    font=dict(family='Arial', size=11),
    coloraxis_showscale=False,
    margin=dict(l=80, r=30, t=90, b=70)
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='black')))
fig.show()

---
<a id="sec3"></a>
### Pre-processing & Feature Engineering

Raw features are encoded, standardised, and partitioned prior to model training.
Two interaction terms are further derived to capture nutrient balance and
combined climatic effects.

#### Encoding, Scaling, and Train–Test Split

In [12]:
# Encode target labels into integers and apply stratified 70/30 split
label_encoder = LabelEncoder()
df['label_enc'] = label_encoder.fit_transform(df['label'])
class_names = label_encoder.classes_

X = df[features].values
y = df['label_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Standardise features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

y_test_categories = [crop_to_cat[class_names[i]] for i in y_test]

print("=" * 55)
print("DATA PREPROCESSING SUMMARY")
print("=" * 55)
print(f"  Training samples        : {X_train.shape[0]:,}")
print(f"  Testing samples         : {X_test.shape[0]:,}")
print(f"  Number of features      : {X_train.shape[1]}")
print(f"  Number of crop classes  : {len(class_names)}")
print(f"  Number of categories    : {len(cat_names)}")
print(f"  Scaling method          : StandardScaler (μ=0, σ=1)")
print(f"  Train-test split ratio  : 70/30 (stratified)")
print(f"  Random seed             : {SEED}")

DATA PREPROCESSING SUMMARY
  Training samples        : 1,540
  Testing samples         : 660
  Number of features      : 7
  Number of crop classes  : 22
  Number of categories    : 6
  Scaling method          : StandardScaler (μ=0, σ=1)
  Train-test split ratio  : 70/30 (stratified)
  Random seed             : 42


#### Feature Engineering

In [13]:
# Construct two composite interaction features from domain knowledge
df['NPK_ratio']           = df['N'] / (df['P'] + df['K'] + 1e-6)
df['temp_humidity_index'] = (df['temperature'] * df['humidity']) / 100.0

print("Engineered Features:")
print("  NPK_ratio           — Nitrogen relative to combined P and K nutrient balance")
print("  temp_humidity_index — Composite index of temperature–humidity interaction")
print()
display(df[['NPK_ratio', 'temp_humidity_index']].describe().round(4))

Engineered Features:
  NPK_ratio           — Nitrogen relative to combined P and K nutrient balance
  temp_humidity_index — Composite index of temperature–humidity interaction



,NPK_ratio,temp_humidity_index
count,2200.0000,2200.0000
mean,0.6801,18.5423
std,0.5785,6.9937
min,0.0000,2.4761
25%,0.2179,14.7956
50%,0.4916,19.2788
75%,1.0251,22.5575
max,2.7800,40.7316


---
<a id="sec4"></a>
### Multi-Objective Model Training

Four classifiers are instantiated, each occupying a distinct position on the
**accuracy–interpretability Pareto frontier**:

| Model | Paradigm | XAI Suitability |
|:------|:---------|:----------------|
| **Random Forest** | Ensemble bagging | High — tree-based, SHAP-compatible |
| **XGBoost** | Gradient boosting | High — SHAP TreeExplainer native |
| **Support Vector Machine** | Kernel margin maximisation | Moderate — permutation-based attribution |
| **Artificial Neural Network** | Deep learning (MLP) | Low — black-box, requires proxy methods |

In [14]:
# Instantiate all candidate classifiers with carefully tuned hyperparameters
model_defs = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_features='sqrt',
        class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=SEED, verbosity=0
    ),
    'Support Vector Machine': SVC(
        kernel='rbf', C=10, gamma='scale',
        class_weight='balanced', probability=True, random_state=SEED
    ),
    'Artificial Neural Network': MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
        alpha=1e-4, batch_size=32, learning_rate='adaptive',
        max_iter=500, early_stopping=True, validation_fraction=0.1,
        random_state=SEED
    )
}

trained_models  = {}
training_times  = {}
model_names     = list(model_defs.keys())

print("Model training in progress...\n")

for name, model in model_defs.items():
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start_time
    trained_models[name] = model
    training_times[name] = elapsed
    print(f"  {name:<28} completed in {elapsed:.2f}s")

print("\nAll models trained successfully.")

Model training in progress...

  Random Forest                completed in 0.64s
  XGBoost                      completed in 4.00s
  Support Vector Machine       completed in 0.35s
  Artificial Neural Network    completed in 2.28s

All models trained successfully.


---
<a id="sec5"></a>
### Comparative Performance Evaluation

Each model is assessed on five standard classification metrics computed on the
held-out test set, followed by a 5-fold stratified cross-validation to evaluate
generalisation stability.

#### Test-Set Metrics

In [15]:
# Compute accuracy, precision, recall, F1, and macro AUC-ROC for all models
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)
    y_bin  = label_binarize(y_test, classes=np.arange(len(class_names)))

    results[name] = {
        'Accuracy':          round(accuracy_score(y_test, y_pred), 4),
        'Precision':         round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'Recall':            round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score':          round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'AUC-ROC':           round(roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro'), 4),
        'Training Time (s)': round(training_times[name], 2),
        'y_pred':            y_pred,
        'y_prob':            y_prob,
        'y_pred_cats':       [crop_to_cat[class_names[i]] for i in y_pred]
    }

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Training Time (s)']
metrics_df  = pd.DataFrame(
    {model: {m: results[model][m] for m in metric_cols} for model in results}
).T

print("=" * 70)
print("COMPARATIVE MODEL PERFORMANCE (TEST SET)")
print("=" * 70)

styled_table = (
    metrics_df.style
    .format("{:.4f}")
    .background_gradient(cmap='Greens', subset=['Accuracy','Precision','Recall','F1-Score','AUC-ROC'])
    .background_gradient(cmap='Reds_r',  subset=['Training Time (s)'])
    .highlight_max(subset=['Accuracy','F1-Score','AUC-ROC'], color='#c8e6c9')
    .highlight_min(subset=['Training Time (s)'],             color='#c8e6c9')
    .set_properties(**{'color':'black','font-family':'Arial','font-size':'11pt','text-align':'center'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-family','Arial'),('font-size','12pt'),
                  ('text-align','center'),('font-weight','bold'),('color','black')]
    }])
)
display(styled_table)

TABLE 1 — COMPARATIVE MODEL PERFORMANCE (TEST SET)


,Accuracy,Precision,Recall,F1-Score,AUC-ROC,Training Time (s)
Random Forest,0.9939,0.9942,0.9939,0.9939,1.0000,0.6400
XGBoost,0.9894,0.9900,0.9894,0.9894,1.0000,4.0000
Support Vector Machine,0.9924,0.9929,0.9924,0.9924,1.0000,0.3500
Artificial Neural Network,0.9864,0.9870,0.9864,0.9863,0.9999,2.2800


#### Performance Comparison

In [16]:
# Grouped bar chart comparing all five performance metrics across models
fig = go.Figure()
metric_display = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']

for name in model_names:
    color  = MODEL_COLORS.get(name, '#888888')
    values = [results[name][m] for m in metric_display]

    fig.add_trace(go.Bar(
        name=name, x=metric_display, y=values,
        marker=dict(color=color, line=dict(width=0.6, color='black')),
        text=[f'{v:.4f}' for v in values],
        textposition='outside',
        textfont=dict(size=11, color='black'),
        hovertemplate=f'<b>{name}</b><br>%{{x}}: %{{y:.4f}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text=(
            "<b>Comparative Performance of Machine Learning Models</b><br>"
            "<span style='font-size:12px'>Evaluation across Accuracy, Precision, Recall, F1-Score, and AUC-ROC</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    barmode='group', template='simple_white',
    font=dict(family='Arial', size=11, color='black'),
    xaxis=dict(title='Evaluation Metrics', tickfont=dict(size=11), title_font=dict(size=13)),
    yaxis=dict(title='Performance Score', range=[0.88, 1.02],
               tickfont=dict(size=11), title_font=dict(size=13)),
    legend=dict(title='Model', orientation='h', x=0.5, xanchor='center', y=-0.22),
    margin=dict(l=60, r=40, t=95, b=80),
    height=520, width=850
)
fig.show()

#### Five-Fold Stratified Cross-Validation

In [17]:
# Assess generalisation stability via 5-fold stratified cross-validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

print("Running 5-Fold Stratified Cross-Validation...\n")
cv_results = {}
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, model in trained_models.items():
    print(f"  Evaluating {name}...")
    scores = cross_val_score(model, X_train_scaled, y_train,
                              cv=kf, scoring='accuracy', n_jobs=1)
    cv_results[name] = scores
    print(f"    Mean Accuracy : {scores.mean():.4f}")
    print(f"    Std Dev       : {scores.std():.4f}")
    print("  " + "-" * 38)

Running 5-Fold Stratified Cross-Validation...

  Evaluating Random Forest...
    Mean Accuracy : 0.9948
    Std Dev       : 0.0053
  --------------------------------------
  Evaluating XGBoost...
    Mean Accuracy : 0.9929
    Std Dev       : 0.0038
  --------------------------------------
  Evaluating Support Vector Machine...
    Mean Accuracy : 0.9792
    Std Dev       : 0.0078
  --------------------------------------
  Evaluating Artificial Neural Network...
    Mean Accuracy : 0.9623
    Std Dev       : 0.0180
  --------------------------------------


#### Cross-Validation Box Plots

In [18]:
# Box plots of fold-level accuracy scores with individual data points overlaid
fig = go.Figure()

for name in model_names:
    scores = cv_results[name]
    color  = MODEL_COLORS.get(name, '#888888')

    fig.add_trace(go.Box(
        y=scores, name=name,
        boxmean=True, boxpoints='all', jitter=0.35, pointpos=0,
        marker=dict(color=color, size=7, line=dict(width=0.5, color='black')),
        line=dict(width=1.2),
        hovertemplate=f'<b>{name}</b><br>Fold Score: %{{y:.4f}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text=(
            "<b>Cross-Validation Performance Stability</b><br>"
            "<span style='font-size:12px'>5-Fold accuracy distribution across machine learning models</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white', font=dict(family='Arial', size=11, color='black'),
    yaxis=dict(title='Cross-Validation Accuracy', tickformat='.3f', range=[0.85, 1.02]),
    xaxis=dict(title='Machine Learning Models'),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.20),
    margin=dict(l=60, r=40, t=95, b=80),
    height=500, width=850
)
fig.show()

#### Category-Level Confusion Matrices

In [19]:
# Derive category-level prediction labels for all models
y_test_cats = [crop_to_cat[class_names[i]] for i in y_test]

for name in model_names:
    results[name]['y_pred_cats'] = [
        crop_to_cat[class_names[i]] for i in results[name]['y_pred']
    ]

In [20]:
# 2×2 subplot grid of confusion matrices at the agronomic category level
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f"<b>{name}</b><br><span style='font-size:10px'>Accuracy = {results[name]['Accuracy']:.3f}</span>"
        for name in model_names
    ],
    horizontal_spacing=0.20, vertical_spacing=0.22
)

for idx, name in enumerate(model_names):
    r  = idx // 2 + 1
    c  = idx % 2 + 1
    cm = confusion_matrix(y_test_cats, results[name]['y_pred_cats'], labels=cat_names)

    fig.add_trace(
        go.Heatmap(
            z=cm, x=cat_names, y=cat_names, colorscale='Blues',
            text=cm, texttemplate='%{text}', textfont=dict(size=13, color='black'),
            showscale=True if idx == 3 else False,
            colorbar=dict(title='Count', thickness=16, len=0.70, x=1.06),
            hovertemplate=(
                f'<b>{name}</b><br>True: %{{y}}<br>Predicted: %{{x}}<br>'
                'Count: %{z}<extra></extra>'
            )
        ),
        row=r, col=c
    )

fig.update_layout(
    title=dict(
        text=(
            "<b>Category-Level Confusion Matrices</b><br>"
            "<span style='font-size:12px'>Rows = true categories; Columns = predicted categories</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white', font=dict(family='Arial', size=11, color='black'),
    height=820, width=1080, margin=dict(l=80, r=120, t=150, b=80)
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(title_text='Predicted Category', row=i, col=j,
                         tickangle=-25, tickfont=dict(size=11))
        fig.update_yaxes(title_text='True Category', row=i, col=j,
                         tickfont=dict(size=11))
fig.show()

#### Per-Category F1-Score Heatmap

In [21]:
# Compute per-category weighted F1 scores for all models
per_category_f1 = {}

for model_name in model_names:
    report = classification_report(
        y_test_cats, results[model_name]['y_pred_cats'],
        labels=cat_names, output_dict=True, zero_division=0
    )
    per_category_f1[model_name] = {
        cat: report[cat]['f1-score'] for cat in cat_names
    }

f1_matrix = pd.DataFrame(per_category_f1).T

fig = go.Figure(go.Heatmap(
    z=f1_matrix.values, x=cat_names, y=model_names,
    text=f1_matrix.round(4).values,
    texttemplate='<b>%{text}</b>', textfont=dict(size=14),
    colorscale='RdYlGn', zmin=0.95, zmax=1.00,
    colorbar=dict(title='F1-Score', thickness=16, len=0.85),
    hovertemplate='<b>Model:</b> %{y}<br><b>Category:</b> %{x}<br><b>F1:</b> %{z:.4f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Per-Category F1-Score Heatmap</b><br>"
            "<sup>Comparative classification performance across agronomic crop categories</sup>"
        ),
        x=0.5, font=dict(size=18)
    ),
    template=PLOTLY_TEMPLATE, height=360,
    xaxis=dict(title='Crop Category', tickfont=dict(size=11)),
    yaxis=dict(title='Machine Learning Model', tickfont=dict(size=11)),
    margin=dict(l=80, r=60, t=90, b=60)
)
fig.show()

---
<a id="sec6"></a>
## Global Explainability Analysis (SHAP)

SHAP (SHapley Additive exPlanations) grounds feature attribution in cooperative
game theory, offering both globally consistent and locally accurate explanations.
TreeExplainer is applied to tree-based models; permutation importance is used
as a model-agnostic surrogate for SVM and ANN.

#### SHAP Values — Random Forest

In [22]:
# Compute SHAP values using the exact TreeExplainer for Random Forest
print("Computing SHAP values for Random Forest...")
rf_model     = trained_models['Random Forest']
explainer_rf = shap.TreeExplainer(rf_model)
sv_rf_raw    = explainer_rf.shap_values(X_test_scaled)

sv_rf = np.array(sv_rf_raw)
if sv_rf.ndim == 2:
    sv_rf = sv_rf[:, :, np.newaxis]
elif sv_rf.ndim == 3 and sv_rf.shape[0] != X_test_scaled.shape[0]:
    sv_rf = sv_rf.transpose(1, 2, 0)  # (classes, samples, features) → (samples, features, classes)

shap_abs_rf         = np.abs(sv_rf).mean(axis=(0, 2))
shap_importance_rf  = pd.Series(shap_abs_rf, index=features).sort_values(ascending=False)

print(f"  SHAP array shape : {sv_rf.shape}\n")
print("Global Feature Importance (Random Forest — SHAP):")
for feat, val in shap_importance_rf.items():
    print(f"  {feature_labels[feat]:<25}: {val:.4f}")

Computing SHAP values for Random Forest...
  SHAP array shape : (660, 7, 22)

Global Feature Importance (Random Forest — SHAP):
  Humidity (%)             : 0.0316
  Potassium (K)            : 0.0267
  Nitrogen (N)             : 0.0261
  Rainfall (mm)            : 0.0254
  Phosphorus (P)           : 0.0245
  Temperature (°C)         : 0.0107
  Soil pH                  : 0.0053


#### SHAP Values — XGBoost

In [23]:
# Compute SHAP values using TreeExplainer for XGBoost
print("Computing SHAP values for XGBoost...")
xgb_model     = trained_models['XGBoost']
explainer_xgb = shap.TreeExplainer(xgb_model)
sv_xgb_raw    = explainer_xgb.shap_values(X_test_scaled)

sv_xgb = np.array(sv_xgb_raw)
if sv_xgb.ndim == 2:
    sv_xgb = sv_xgb[:, :, np.newaxis]
elif sv_xgb.ndim == 3 and sv_xgb.shape[0] != X_test_scaled.shape[0]:
    sv_xgb = sv_xgb.transpose(1, 2, 0)

shap_abs_xgb         = np.abs(sv_xgb).mean(axis=(0, 2))
shap_importance_xgb  = pd.Series(shap_abs_xgb, index=features).sort_values(ascending=False)

print(f"  SHAP array shape : {sv_xgb.shape}")

Computing SHAP values for XGBoost...
  SHAP array shape : (660, 7, 22)


#### Global SHAP Importance — Side-by-Side Comparison

In [24]:
# Horizontal bar charts comparing SHAP global importance for RF and XGBoost
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Random Forest — SHAP</b>', '<b>XGBoost — SHAP</b>'),
    horizontal_spacing=0.14
)

for col_i, (imp, palette) in enumerate([
    (shap_importance_rf,  px.colors.sequential.Teal),
    (shap_importance_xgb, px.colors.sequential.Purp)
]):
    sorted_imp = imp.sort_values()
    n = len(sorted_imp)
    colors = [palette[int(i * (len(palette)-1) / (n-1))] for i in range(n)]

    fig.add_trace(
        go.Bar(
            x=sorted_imp.values,
            y=[feature_labels[f] for f in sorted_imp.index],
            orientation='h',
            marker=dict(color=colors),
            text=[f'{v:.4f}' for v in sorted_imp.values],
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Mean |SHAP|: %{x:.4f}<extra></extra>'
        ),
        row=1, col=col_i+1
    )

fig.update_layout(
    title=dict(
        text='<b>SHAP Global Feature Importance (RF & XGBoost)</b>',
        x=0.5, font=dict(size=18)
    ),
    template=PLOTLY_TEMPLATE, height=420, showlegend=False
)
fig.update_xaxes(title_text='Mean |SHAP Value|')
fig.show()

#### Permutation Importance — SVM & ANN

In [25]:
# Permutation importance as a model-agnostic attribution method for SVM and ANN
from sklearn.inspection import permutation_importance

print("Computing permutation importance for Support Vector Machine...")
perm_svm = permutation_importance(
    trained_models['Support Vector Machine'], X_test_scaled, y_test,
    n_repeats=10, random_state=SEED, n_jobs=1
)
print("  Completed.\n")

print("Computing permutation importance for Artificial Neural Network...")
perm_ann = permutation_importance(
    trained_models['Artificial Neural Network'], X_test_scaled, y_test,
    n_repeats=10, random_state=SEED, n_jobs=1
)
print("  Completed.")

Computing permutation importance for Support Vector Machine...
  Completed.

Computing permutation importance for Artificial Neural Network...
  Completed.


#### Consensus Feature Importance — All Models

In [26]:
import textwrap
import pandas as pd
import plotly.graph_objects as go

# Wrap long feature labels to avoid overlap
wrapped_index = [
    "<br>".join(textwrap.wrap(label, width=12))
    for label in [feature_labels[f] for f in features]
]

# Normalised importance scores unified across all four models
importance_df = pd.DataFrame(
    {
        'RF (SHAP)':   shap_importance_rf.values,
        'XGB (SHAP)':  shap_importance_xgb.loc[features].values,
        'SVM (Perm)':  perm_svm.importances_mean,
        'ANN (Perm)':  perm_ann.importances_mean,
    },
    index=wrapped_index
)

# Column-wise normalisation
importance_norm = importance_df.div(importance_df.abs().sum(axis=0), axis=1)

# Colors
method_colors = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

# Create figure
fig = go.Figure()

for method, color in zip(importance_norm.columns, method_colors):
    fig.add_trace(go.Bar(
        name=method,
        x=importance_norm.index,
        y=importance_norm[method],
        marker=dict(color=color, line=dict(width=0.6, color='black')),
        hovertemplate=(
            f'<b>Method:</b> {method}<br>'
            '<b>Feature:</b> %{x}<br>'
            '<b>Normalised Importance:</b> %{y:.4f}<extra></extra>'
        )
    ))

# Layout adjustments
fig.update_layout(
    title=dict(
        text=(
            "<b>Consensus Feature Importance Across Models</b><br>"
            "<span style='font-size:12px'>SHAP (RF, XGB) vs Permutation (SVM, ANN)</span>"
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=18)
    ),

    template='simple_white',
    barmode='group',
    font=dict(size=11),
    height=550,
    width=950,

    xaxis=dict(
        title='Input Features',
        tickangle=0
    ),

    yaxis=dict(
        title='Normalised Importance Score'
    ),

    legend=dict(
        title='Method',
        orientation='h',
        x=0.5,
        xanchor='center',
        y=-0.35 
    ),

    margin=dict(
        l=80,
        r=40,
        t=100,
        b=160 
    )
)

fig.show()

---
<a id="sec7"></a>
### Local Explainability Analysis (LIME)

LIME (Local Interpretable Model-agnostic Explanations) fits a locally faithful linear
surrogate around individual predictions, generating actionable, human-readable
explanations for each crop recommendation. This is particularly relevant for
farmer-facing advisory systems where per-instance justification builds trust.

#### LIME Explainer Initialisation

In [27]:
# Initialise a LIME tabular explainer fitted on the scaled training distribution
lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=[feature_labels[f] for f in features],
    class_names=class_names,
    mode='classification',
    discretize_continuous=True,
    random_state=SEED
)

print("LIME explainer ready.")
print(f"  Training instances : {X_train_scaled.shape[0]:,}")
print(f"  Classes            : {len(class_names)}")

LIME explainer ready.
  Training instances : 1,540
  Classes            : 22


#### Instance-Level LIME Explanations (XGBoost)

In [28]:
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Select one test instance per category (first match)
sample_indices = []
for cat in cat_names:
    idxs = [i for i, lbl in enumerate(y_test_categories) if lbl == cat]
    if idxs:
        sample_indices.append((idxs[0], cat, class_names[y_test[idxs[0]]]))

n_samples = len(sample_indices)

# 3-column layout
n_cols = 3
n_rows = math.ceil(n_samples / n_cols)

# Subplot titles
subplot_titles = [
    f"<b>{crop}</b><br><span style='font-size:10px'>Category: {grp}</span>"
    for _, grp, crop in sample_indices
]

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=subplot_titles,

    # Increase spacing between columns and rows
    horizontal_spacing=0.12,  
    vertical_spacing=0.20      
)

# Loop through samples
for plot_i, (idx, grp, crop) in enumerate(sample_indices):
    exp = lime_explainer.explain_instance(
        X_test_scaled[idx],
        trained_models['XGBoost'].predict_proba,
        num_features=7,
        top_labels=1
    )

    top_lbl = exp.available_labels()[0]
    exp_list = exp.as_list(label=top_lbl)

    feat_names = [e[0] for e in exp_list]
    feat_vals  = [e[1] for e in exp_list]

    colors = ['#2ca02c' if v > 0 else '#d62728' for v in feat_vals]

    r = plot_i // n_cols + 1
    c = plot_i % n_cols + 1

    fig.add_trace(
        go.Bar(
            x=feat_vals[::-1],
            y=feat_names[::-1],
            orientation='h',
            marker=dict(
                color=colors[::-1],
                line=dict(color='white', width=0.6)
            ),
            hovertemplate='<b>%{y}</b><br>LIME weight: %{x:.4f}<extra></extra>'
        ),
        row=r, col=c
    )

    fig.add_vline(
        x=0,
        line_width=1.2,
        line_color='#2C3E50',
        row=r,
        col=c
    )

# Layout
fig.update_layout(
    title=dict(
        text=(
            "<b>LIME Local Explanations for Crop Prediction (XGBoost)</b><br>"
            "<span style='font-size:12px'>Green: Positive contribution | Red: Negative contribution</span>"
        ),
        x=0.5,
        font=dict(size=18)
    ),

    template='simple_white',

    height=340 * n_rows,  
    width=1150,

    showlegend=False,

    margin=dict(
        l=60,
        r=40,
        t=140, 
        b=70
    )
)

# Axis formatting
for i in range(1, n_rows + 1):
    for j in range(1, n_cols + 1):
        fig.update_xaxes(
            title_text='LIME Weight',
            showgrid=True,
            zeroline=False,
            row=i, col=j
        )
        fig.update_yaxes(
            showgrid=False,
            row=i, col=j
        )

fig.show()

In [29]:
# Verify LIME consistency by comparing explanations for the same instance across all models
if sample_indices:
    idx_ref, grp_ref, crop_ref = sample_indices[0]

    fig = make_subplots(
        rows=1, cols=4,
        subplot_titles=[f'<b>{n}</b>' for n in model_names],
        horizontal_spacing=0.06
    )

    for col_i, name in enumerate(model_names):
        exp_m   = lime_explainer.explain_instance(
            X_test_scaled[idx_ref], trained_models[name].predict_proba,
            num_features=7, top_labels=1
        )
        top_lbl = exp_m.available_labels()[0]
        el      = exp_m.as_list(label=top_lbl)
        fnames  = [e[0] for e in el]
        fvals   = [e[1] for e in el]
        cols    = ['#27AE60' if v > 0 else '#E74C3C' for v in fvals]

        fig.add_trace(
            go.Bar(x=fvals[::-1], y=fnames[::-1], orientation='h',
                   marker=dict(color=cols[::-1]),
                   hovertemplate=f'<b>{name}</b><br>%{{y}}: %{{x:.4f}}<extra></extra>'),
            row=1, col=col_i+1
        )
        fig.add_vline(x=0, line_width=1.2, line_color='#2C3E50', row=1, col=col_i+1)

    fig.update_layout(
        title=dict(
            text=f'<b>LIME Stability: Crop "{crop_ref}" Explained Across All Models</b>',
            x=0.5, font=dict(size=16)
        ),
        template=PLOTLY_TEMPLATE, height=420, showlegend=False
    )
    fig.show()

---
<a id="sec8"></a>
### Multi-Objective Trade-off Analysis

This section constitutes the core thesis contribution. A structured XAI rubric
quantifies model transparency across five explainability dimensions, which is
then combined with predictive performance into a **Multi-Objective Composite Score (MOCS)**
to identify the optimal model on the accuracy–interpretability Pareto frontier.

$$\text{MOCS} = 0.40 \cdot \tilde{F}_1 + 0.30 \cdot \widetilde{\text{AUC}} + 0.30 \cdot \widetilde{\text{XAI}}$$

where $\tilde{\cdot}$ denotes min-max normalisation.

#### XAI Explainability Rubric

In [30]:
# Structured rubric scoring five explainability dimensions (0–100) per model
explainability_rubric = {
    'Random Forest': {'Structural Transparency': 55, 'Feature Attribution': 90,
                      'Local Explainability': 85, 'Computational XAI Cost': 80,
                      'Human Interpretability': 60},
    'XGBoost':       {'Structural Transparency': 45, 'Feature Attribution': 95,
                      'Local Explainability': 90, 'Computational XAI Cost': 85,
                      'Human Interpretability': 50},
    'SVM':           {'Structural Transparency': 30, 'Feature Attribution': 50,
                      'Local Explainability': 60, 'Computational XAI Cost': 40,
                      'Human Interpretability': 25},
    'ANN':           {'Structural Transparency': 15, 'Feature Attribution': 45,
                      'Local Explainability': 55, 'Computational XAI Cost': 30,
                      'Human Interpretability': 10}
}

xai_weights = {
    'Structural Transparency': 0.20,
    'Feature Attribution':     0.25,
    'Local Explainability':    0.25,
    'Computational XAI Cost':  0.15,
    'Human Interpretability':  0.15
}

# Compute weighted composite XAI score per model
explainability_scores = {
    name: round(sum(dims[d] * xai_weights[d] for d in dims), 2)
    for name, dims in explainability_rubric.items()
}

print("XAI Composite Scores:")
for name, score in explainability_scores.items():
    print(f"  {name:<28}: {score:.2f}/100")

XAI Composite Scores:
  Random Forest               : 75.75/100
  XGBoost                     : 75.50/100
  SVM                         : 43.25/100
  ANN                         : 34.00/100


#### XAI Dimension Profiles

In [31]:
# Radar (spider) chart comparing the explainability profiles of all four models
model_name_map = {
    'Random Forest':            'Random Forest',
    'XGBoost':                  'XGBoost',
    'Support Vector Machine':   'SVM',
    'Artificial Neural Network':'ANN'
}

categories_xai        = list(xai_weights.keys())
categories_xai_closed = categories_xai + [categories_xai[0]]

fig = go.Figure()

for name in model_names:
    key        = model_name_map[name]
    vals       = [explainability_rubric[key][c] for c in categories_xai]
    vals_closed = vals + [vals[0]]

    fig.add_trace(go.Scatterpolar(
        r=vals_closed, theta=categories_xai_closed,
        fill='toself', fillcolor=MODEL_COLORS[key],
        line=dict(color=MODEL_COLORS[key], width=2.5),
        opacity=0.35, name=name,
        hovertemplate=f'<b>{name}</b><br>%{{theta}}: %{{r:.2f}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text=(
            "<b>XAI Dimension Profiles</b><br>"
            "<span style='font-size:12px'>Comparative explainability across models</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    polar=dict(
        radialaxis=dict(visible=True, range=[0,100],
                        tickfont=dict(size=9, family='Arial'),
                        gridcolor='lightgrey', gridwidth=1),
        angularaxis=dict(tickfont=dict(size=11, family='Arial'),
                         rotation=90, direction='clockwise')
    ),
    template='simple_white',
    legend=dict(title='<b>Model</b>', orientation='h', x=0.5, xanchor='center',
                y=-0.12, font=dict(size=11)),
    font=dict(family='Arial', size=11),
    height=530, margin=dict(l=80, r=80, t=100, b=100)
)
fig.show()

#### MOCS Calculation & Ranking

In [32]:
# Compute the Multi-Objective Composite Score (MOCS) for each model
model_key_map = {
    'Random Forest':            'Random Forest',
    'XGBoost':                  'XGBoost',
    'Support Vector Machine':   'SVM',
    'Artificial Neural Network':'ANN'
}

f1_arr  = np.array([results[n]['F1-Score'] for n in model_names])
auc_arr = np.array([results[n]['AUC-ROC']  for n in model_names])
xai_arr = np.array([explainability_scores[model_key_map[n]] for n in model_names]) / 100.0

def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-10)

alpha_w, beta_w, gamma_w = 0.40, 0.30, 0.30
mocs = alpha_w*minmax(f1_arr) + beta_w*minmax(auc_arr) + gamma_w*minmax(xai_arr)

mocs_df = pd.DataFrame({
    'Model':     model_names,
    'F1-Score':  f1_arr,
    'AUC-ROC':   auc_arr,
    'XAI Score': (xai_arr*100).round(2),
    'Norm F1':   minmax(f1_arr).round(4),
    'Norm AUC':  minmax(auc_arr).round(4),
    'Norm XAI':  minmax(xai_arr).round(4),
    'MOCS':      mocs.round(4)
}).set_index('Model').sort_values('MOCS', ascending=False)

best_model_name = mocs_df.index[0]

print("MOCS RANKING")
display(
    mocs_df.style
        .background_gradient(cmap='YlGn', subset=['MOCS'])
        .format(precision=4)
        .highlight_max(subset=['MOCS'], color='#c8e6c9')
)
print(f"\n  Optimal model (MOCS): {best_model_name}  ({mocs_df.loc[best_model_name,'MOCS']:.4f})")

MOCS RANKING


,F1-Score,AUC-ROC,XAI Score,Norm F1,Norm AUC,Norm XAI,MOCS
Model,,,,,,,
Random Forest,0.9939,1.0000,75.7500,1.0000,1.0000,1.0000,1.0000
XGBoost,0.9894,1.0000,75.5000,0.4079,1.0000,0.9940,0.7614
Support Vector Machine,0.9924,1.0000,43.2500,0.8026,1.0000,0.2216,0.6875
Artificial Neural Network,0.9863,0.9999,34.0000,0.0000,0.0000,0.0000,0.0000



  Optimal model (MOCS): Random Forest  (1.0000)


#### Predictive Power vs. Transparency

In [33]:
# Pareto bubble chart: XAI Score (x), F1-Score (y), AUC-ROC (bubble size)
color_key_map = {
    'Random Forest':            'Random Forest',
    'XGBoost':                  'XGBoost',
    'Support Vector Machine':   'SVM',
    'Artificial Neural Network':'ANN'
}

bubble_df = mocs_df.reset_index().rename(columns={'index':'Model'})
bubble_df['F1-Score'] = bubble_df['F1-Score'] * 100

fig = go.Figure()

# Highlight the preferred (high accuracy + high explainability) region
fig.add_shape(type='rect',
    x0=60, x1=105, y0=98.5, y1=100.4,
    fillcolor='rgba(46,204,113,0.08)',
    line=dict(color='rgba(46,204,113,0.4)', dash='dot', width=1.5)
)
fig.add_annotation(x=82, y=100.35,
    text='<b>Preferred Region</b><br>(High Accuracy + High Explainability)',
    showarrow=False, font=dict(size=10, color='#27AE60'),
    bgcolor='rgba(255,255,255,0.7)'
)

for _, row in bubble_df.iterrows():
    name = row['Model']
    fig.add_trace(go.Scatter(
        x=[row['XAI Score']], y=[row['F1-Score']],
        mode='markers+text',
        marker=dict(
            size=row['AUC-ROC']*120,
            color=MODEL_COLORS[color_key_map[name]],
            line=dict(color='white', width=2.5), opacity=0.88
        ),
        text=[f'<b>{name}</b>'], textposition='middle right',
        textfont=dict(size=11, color='black'), name=name,
        hovertemplate=(
            f'<b>{name}</b><br>'
            f'XAI Score: {row["XAI Score"]:.2f}<br>'
            f'F1-Score: {row["F1-Score"]:.2f}%<br>'
            f'AUC-ROC: {row["AUC-ROC"]:.4f}<br>'
            f'MOCS: {row["MOCS"]:.4f}<extra></extra>'
        )
    ))

fig.update_layout(
    title=dict(
        text=(
            '<b>Pareto Trade-off: Predictive Power vs. Transparency</b><br>'
            '<sup>Bubble size proportional to AUC-ROC | Upper-right = optimal zone</sup>'
        ),
        x=0.5, font=dict(size=17)
    ),
    template=PLOTLY_TEMPLATE,
    xaxis=dict(title='XAI Explainability Score (0–100)', range=[0, 110]),
    yaxis=dict(title='Weighted F1-Score (%)', range=[96, 100.5]),
    showlegend=False, height=520,
    margin=dict(l=80, r=80, t=100, b=80)
)
fig.show()

---
<a id="sec9"></a>
### Statistical Significance Testing

Non-parametric tests are employed to validate that observed performance differences
are statistically meaningful and not attributable to random variation across folds.

#### Friedman Test (Global Comparison)

In [34]:
# Friedman chi-squared test across all four models' CV fold distributions
cv_arrays = [cv_results[n] for n in model_names]
stat_f, p_val_f = stats.friedmanchisquare(*cv_arrays)

print("=" * 60)
print("FRIEDMAN TEST  (non-parametric, k=4 models)")
print("=" * 60)
print(f"  Chi-squared : {stat_f:.4f}")
print(f"  p-value     : {p_val_f:.6f}")
print(f"  Decision    : {'REJECT H₀  — significant differences exist' if p_val_f < 0.05 else 'FAIL TO REJECT H₀'}")

FRIEDMAN TEST  (non-parametric, k=4 models)
  Chi-squared : 14.3478
  p-value     : 0.002468
  Decision    : REJECT H₀  — significant differences exist


#### Pairwise Wilcoxon Tests

In [35]:
import plotly.graph_objects as go
from itertools import combinations
from scipy import stats
import numpy as np
import pandas as pd

# -------------------------------
# Pairwise Wilcoxon Tests
# -------------------------------
pairs = list(combinations(model_names, 2))

pval_mat = pd.DataFrame(
    np.ones((len(model_names), len(model_names))),
    index=model_names,
    columns=model_names
)

results = []

for m1, m2 in pairs:
    try:
        stat, p_val = stats.wilcoxon(cv_results[m1], cv_results[m2])
    except ValueError:
        stat, p_val = np.nan, 1.0

    # Fill symmetric matrix
    pval_mat.loc[m1, m2] = p_val
    pval_mat.loc[m2, m1] = p_val

    results.append({
        "Model A": m1,
        "Model B": m2,
        "p-value": round(p_val, 6),
        "Significant (α=0.05)": "Yes" if p_val < 0.05 else "No"
    })

print("Pairwise Wilcoxon Signed-Rank Test Results")
display(pd.DataFrame(results))


mask = np.eye(len(model_names), dtype=bool)

z = pval_mat.values.astype(float)
z[mask] = np.nan  

# Add significance marker (*)
text_vals = np.where(
    mask,
    "",
    np.where(
        pval_mat.values < 0.05,
        pval_mat.round(4).astype(str) + " *",
        pval_mat.round(4).astype(str)
    )
)

fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=z,
    x=model_names,
    y=model_names,

    colorscale=[
        [0.0, '#1a9850'],   
        [0.5, '#fee08b'],   
        [1.0, '#d73027']    
    ],

    zmin=0,
    zmax=0.10,

    text=text_vals,
    texttemplate='%{text}',
    textfont=dict(size=13, color='black'),

    colorbar=dict(
        title='p-value',
        thickness=16,
        tickfont=dict(size=11)
    ),

    hovertemplate='<b>%{y} vs %{x}</b><br>p-value: %{z:.5f}<extra></extra>'
))


fig.update_layout(
    title=dict(
        text=(
            "<b>Pairwise Model Comparison Using Wilcoxon Signed-Rank Test</b><br>"
            "<span style='font-size:12px'>* indicates statistical significance at α = 0.05</span>"
        ),
        x=0.5,
        font=dict(size=18)
    ),

    template='simple_white',

    height=550,
    width=700,

    margin=dict(
        l=100,
        r=60,
        t=130,
        b=150   
    )
)

# Axis formatting
fig.update_xaxes(
    title_text="Model",
    tickangle=-20,
    tickfont=dict(size=12)
)

fig.update_yaxes(
    title_text="Model",
    tickfont=dict(size=12),
    autorange='reversed'
)

fig.add_annotation(
    text="* Significant at α = 0.05",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.30,  
    showarrow=False,
    font=dict(size=11, color="gray")
)

fig.show()

Pairwise Wilcoxon Signed-Rank Test Results


,Model A,Model B,p-value,Significant (α=0.05)
0,Random Forest,XGBoost,0.5000,No
1,Random Forest,Support Vector Machine,0.0625,No
2,Random Forest,Artificial Neural Network,0.0625,No
3,XGBoost,Support Vector Machine,0.0625,No
4,XGBoost,Artificial Neural Network,0.0625,No
5,Support Vector Machine,Artificial Neural Network,0.1250,No


#### Cohen's *d* Effect Size

In [36]:
# Cohen's d quantifies the practical magnitude of differences between model pairs
def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / (
        np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2) + 1e-10
    )

eff_rows = []
for m1, m2 in pairs:
    d   = cohens_d(cv_results[m1], cv_results[m2])
    mag = ('Negligible' if abs(d) < 0.2 else
           'Small'      if abs(d) < 0.5 else
           'Medium'     if abs(d) < 0.8 else 'Large')
    eff_rows.append({'Model A': m1, 'Model B': m2,
                     "Cohen's d": round(d, 4), 'Magnitude': mag})

print("Effect Size (Cohen's d)")
display(pd.DataFrame(eff_rows))

Effect Size (Cohen's d)


,Model A,Model B,Cohen's d,Magnitude
0,Random Forest,XGBoost,0.3795,Small
1,Random Forest,Support Vector Machine,2.0850,Large
2,Random Forest,Artificial Neural Network,2.1926,Large
3,XGBoost,Support Vector Machine,1.9799,Large
4,XGBoost,Artificial Neural Network,2.1019,Large
5,Support Vector Machine,Artificial Neural Network,1.0890,Large
